In [1]:
import pandas as pd
import numpy as np
import os
import json

In [2]:
clustered_df = pd.read_csv(
    "../Data/attributes/structured_component_dataset_v1_clustered.csv"
)

print("Clustered dataset shape:", clustered_df.shape)
clustered_df.head()

Clustered dataset shape: (155, 27)


,image,component_id,class_id,component,confidence,x1,y1,x2,y2,width,...,normalized_width,normalized_height,brightness,contrast,min_intensity,max_intensity,detected_text,has_text,text_count,cluster
0,10002.png,0,2,CheckedTextView,0.502286,0.821686,82.077820,84.756989,284.991638,83.935303,...,0.058288,0.079263,179.031471,44.581621,50,255,NaN,False,0,0
1,10010.png,0,2,CheckedTextView,0.630227,0.432907,83.921669,83.684662,280.505280,83.251755,...,0.057814,0.076790,158.619350,33.390921,146,255,NaN,False,0,0
2,10010.png,1,2,CheckedTextView,0.427453,23.193726,80.703644,1363.871338,285.599701,1340.677612,...,0.931026,0.080038,186.758748,39.678008,146,255,NaN,False,0,1
3,10010.png,2,3,EditText,0.278310,0.000000,80.900490,82.139648,278.541290,82.139648,...,0.057041,0.077203,159.714400,35.140015,146,255,NaN,False,0,0
4,10031.png,0,2,CheckedTextView,0.597760,1.192478,86.552521,83.562469,278.167755,82.369991,...,0.057201,0.074850,89.039634,28.920227,84,255,NaN,False,0,0


In [3]:
cooccurrence_df = pd.read_csv("../Data/patterns/component_cooccurrence.csv")

spatial_df = pd.read_csv("../Data/patterns/spatial_relationships.csv")

patterns_df = pd.read_csv("../Data/patterns/component_spatial_patterns.csv")

print("Co-occurrence:", cooccurrence_df.shape)
print("Spatial:", spatial_df.shape)
print("Patterns:", patterns_df.shape)

Co-occurrence: (6, 3)
Spatial: (120, 8)
Patterns: (16, 5)


In [4]:
os.makedirs("../Data/rag", exist_ok=True)

In [5]:
component_records = []

for component, group in clustered_df.groupby("component"):

    component_records.append(
        {
            "record_type": "component_profile",
            "title": f"UI Component: {component}",
            "text": (
                f"The UI component type {component} appears "
                f"{len(group)} times in the analyzed RICO screenshots. "
                f"Its average width is {group['width'].mean():.2f} pixels, "
                f"average height is {group['height'].mean():.2f} pixels, "
                f"average area is {group['area'].mean():.2f} square pixels, "
                f"and average aspect ratio is {group['aspect_ratio'].mean():.2f}. "
                f"The average detection confidence is "
                f"{group['confidence'].mean():.3f}."
            ),
            "source": "structured_component_dataset_v1_clustered.csv",
        }
    )

component_records_df = pd.DataFrame(component_records)

component_records_df

,record_type,title,text,source
0,component_profile,UI Component: Button,The UI component type Button appears 41 times ...,structured_component_dataset_v1_clustered.csv
1,component_profile,UI Component: CheckBox,The UI component type CheckBox appears 5 times...,structured_component_dataset_v1_clustered.csv
2,component_profile,UI Component: CheckedTextView,The UI component type CheckedTextView appears ...,structured_component_dataset_v1_clustered.csv
3,component_profile,UI Component: EditText,The UI component type EditText appears 38 time...,structured_component_dataset_v1_clustered.csv


In [6]:
cluster_records = []

for cluster, group in clustered_df.groupby("cluster"):

    component_counts = group["component"].value_counts()
    top_component = component_counts.index[0]
    top_percentage = component_counts.iloc[0] / len(group) * 100

    cluster_records.append(
        {
            "record_type": "cluster_profile",
            "title": f"UI Component Cluster {cluster}",
            "text": (
                f"Cluster {cluster} contains {len(group)} UI components. "
                f"The most common component in this cluster is "
                f"{top_component}, representing {top_percentage:.2f}% "
                f"of the cluster. "
                f"The average width is {group['width'].mean():.2f} pixels, "
                f"average height is {group['height'].mean():.2f} pixels, "
                f"average area is {group['area'].mean():.2f} square pixels, "
                f"and average aspect ratio is "
                f"{group['aspect_ratio'].mean():.2f}."
            ),
            "source": "structured_component_dataset_v1_clustered.csv",
        }
    )

cluster_records_df = pd.DataFrame(cluster_records)

cluster_records_df

,record_type,title,text,source
0,cluster_profile,UI Component Cluster 0,Cluster 0 contains 128 UI components. The most...,structured_component_dataset_v1_clustered.csv
1,cluster_profile,UI Component Cluster 1,Cluster 1 contains 18 UI components. The most ...,structured_component_dataset_v1_clustered.csv
2,cluster_profile,UI Component Cluster 2,Cluster 2 contains 9 UI components. The most c...,structured_component_dataset_v1_clustered.csv


In [7]:
cooccurrence_records = []

for _, row in cooccurrence_df.iterrows():

    component_1 = row["component_1"]
    component_2 = row["component_2"]
    count = row["cooccurrence"]

    if count <= 0:
        continue

    cooccurrence_records.append(
        {
            "record_type": "component_cooccurrence",
            "title": f"{component_1} and {component_2} Co-occurrence",
            "text": (
                f"{component_1} and {component_2} "
                f"occur together in {int(count)} analyzed screenshots. "
                f"This represents a detected co-occurrence pattern "
                f"in the analyzed RICO UI data."
            ),
            "source": "component_cooccurrence.csv",
        }
    )

cooccurrence_records_df = pd.DataFrame(cooccurrence_records)

cooccurrence_records_df.head(10)

,record_type,title,text,source
0,component_cooccurrence,Button and EditText Co-occurrence,Button and EditText occur together in 37 analy...,component_cooccurrence.csv
1,component_cooccurrence,Button and CheckedTextView Co-occurrence,Button and CheckedTextView occur together in 1...,component_cooccurrence.csv
2,component_cooccurrence,CheckedTextView and EditText Co-occurrence,CheckedTextView and EditText occur together in...,component_cooccurrence.csv
3,component_cooccurrence,Button and CheckBox Co-occurrence,Button and CheckBox occur together in 5 analyz...,component_cooccurrence.csv
4,component_cooccurrence,CheckBox and EditText Co-occurrence,CheckBox and EditText occur together in 5 anal...,component_cooccurrence.csv
5,component_cooccurrence,CheckBox and CheckedTextView Co-occurrence,CheckBox and CheckedTextView occur together in...,component_cooccurrence.csv


In [8]:
spatial_records = []

for _, row in patterns_df.iterrows():

    component_1 = row["component_1"]
    component_2 = row["component_2"]
    relationship = row["relationship"]
    frequency = row["frequency"]

    spatial_records.append(
        {
            "record_type": "spatial_relationship",
            "title": (f"{component_1} {relationship} {component_2}"),
            "text": (
                f"A {component_1} is positioned "
                f"{relationship.replace('_', ' ')} "
                f"a {component_2} in the analyzed UI screenshots. "
                f"This spatial relationship was observed "
                f"{int(frequency)} times."
            ),
            "source": "component_spatial_patterns.csv",
        }
    )

spatial_records_df = pd.DataFrame(spatial_records)

spatial_records_df.head(10)

,record_type,title,text,source
0,spatial_relationship,Button above EditText,A Button is positioned above a EditText in the...,component_spatial_patterns.csv
1,spatial_relationship,Button right_of CheckedTextView,A Button is positioned right of a CheckedTextV...,component_spatial_patterns.csv
2,spatial_relationship,CheckedTextView right_of CheckedTextView,A CheckedTextView is positioned right of a Che...,component_spatial_patterns.csv
3,spatial_relationship,CheckedTextView left_of CheckedTextView,A CheckedTextView is positioned left of a Chec...,component_spatial_patterns.csv
4,spatial_relationship,EditText right_of CheckedTextView,A EditText is positioned right of a CheckedTex...,component_spatial_patterns.csv
5,spatial_relationship,CheckedTextView left_of EditText,A CheckedTextView is positioned left of a Edit...,component_spatial_patterns.csv
6,spatial_relationship,Button right_of CheckBox,A Button is positioned right of a CheckBox in ...,component_spatial_patterns.csv
7,spatial_relationship,EditText right_of CheckBox,A EditText is positioned right of a CheckBox i...,component_spatial_patterns.csv
8,spatial_relationship,Button left_of EditText,A Button is positioned left of a EditText in t...,component_spatial_patterns.csv
9,spatial_relationship,CheckBox left_of CheckedTextView,A CheckBox is positioned left of a CheckedText...,component_spatial_patterns.csv


In [9]:
rag_df = pd.concat(
    [
        component_records_df,
        cluster_records_df,
        cooccurrence_records_df,
        spatial_records_df,
    ],
    ignore_index=True,
)

print("Total RAG records:", len(rag_df))
print("\nRecord types:")
print(rag_df["record_type"].value_counts())

Total RAG records: 29

Record types:
record_type
spatial_relationship      16
component_cooccurrence     6
component_profile          4
cluster_profile            3
Name: count, dtype: int64


In [10]:
print("Missing values:")
print(rag_df.isna().sum())

print("\nDuplicate records:")
print(rag_df.duplicated(subset=["text"]).sum())

rag_df.head(10)

Missing values:
record_type    0
title          0
text           0
source         0
dtype: int64

Duplicate records:
0


,record_type,title,text,source
0,component_profile,UI Component: Button,The UI component type Button appears 41 times ...,structured_component_dataset_v1_clustered.csv
1,component_profile,UI Component: CheckBox,The UI component type CheckBox appears 5 times...,structured_component_dataset_v1_clustered.csv
2,component_profile,UI Component: CheckedTextView,The UI component type CheckedTextView appears ...,structured_component_dataset_v1_clustered.csv
3,component_profile,UI Component: EditText,The UI component type EditText appears 38 time...,structured_component_dataset_v1_clustered.csv
4,cluster_profile,UI Component Cluster 0,Cluster 0 contains 128 UI components. The most...,structured_component_dataset_v1_clustered.csv
5,cluster_profile,UI Component Cluster 1,Cluster 1 contains 18 UI components. The most ...,structured_component_dataset_v1_clustered.csv
6,cluster_profile,UI Component Cluster 2,Cluster 2 contains 9 UI components. The most c...,structured_component_dataset_v1_clustered.csv
7,component_cooccurrence,Button and EditText Co-occurrence,Button and EditText occur together in 37 analy...,component_cooccurrence.csv
8,component_cooccurrence,Button and CheckedTextView Co-occurrence,Button and CheckedTextView occur together in 1...,component_cooccurrence.csv
9,component_cooccurrence,CheckedTextView and EditText Co-occurrence,CheckedTextView and EditText occur together in...,component_cooccurrence.csv


In [11]:
rag_df = rag_df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print("Records after deduplication:", len(rag_df))

Records after deduplication: 29


In [12]:
rag_df.insert(0, "record_id", [f"RAG_{i:05d}" for i in range(len(rag_df))])

rag_df.head()

,record_id,record_type,title,text,source
0,RAG_00000,component_profile,UI Component: Button,The UI component type Button appears 41 times ...,structured_component_dataset_v1_clustered.csv
1,RAG_00001,component_profile,UI Component: CheckBox,The UI component type CheckBox appears 5 times...,structured_component_dataset_v1_clustered.csv
2,RAG_00002,component_profile,UI Component: CheckedTextView,The UI component type CheckedTextView appears ...,structured_component_dataset_v1_clustered.csv
3,RAG_00003,component_profile,UI Component: EditText,The UI component type EditText appears 38 time...,structured_component_dataset_v1_clustered.csv
4,RAG_00004,cluster_profile,UI Component Cluster 0,Cluster 0 contains 128 UI components. The most...,structured_component_dataset_v1_clustered.csv


In [13]:
for _, row in rag_df.head(10).iterrows():

    print("=" * 80)
    print("ID:", row["record_id"])
    print("TYPE:", row["record_type"])
    print("TITLE:", row["title"])
    print("TEXT:", row["text"])

ID: RAG_00000
TYPE: component_profile
TITLE: UI Component: Button
TEXT: The UI component type Button appears 41 times in the analyzed RICO screenshots. Its average width is 82.84 pixels, average height is 197.30 pixels, average area is 16351.74 square pixels, and average aspect ratio is 0.42. The average detection confidence is 0.830.
ID: RAG_00001
TYPE: component_profile
TITLE: UI Component: CheckBox
TEXT: The UI component type CheckBox appears 5 times in the analyzed RICO screenshots. Its average width is 559.53 pixels, average height is 197.83 pixels, average area is 110104.36 square pixels, and average aspect ratio is 2.84. The average detection confidence is 0.317.
ID: RAG_00002
TYPE: component_profile
TITLE: UI Component: CheckedTextView
TEXT: The UI component type CheckedTextView appears 71 times in the analyzed RICO screenshots. Its average width is 363.46 pixels, average height is 199.00 pixels, average area is 72760.58 square pixels, and average aspect ratio is 1.82. The aver

In [14]:
rag_csv_path = "../Data/rag/rico_rag_dataset.csv"

rag_df.to_csv(rag_csv_path, index=False)

print("Saved:", rag_csv_path)

Saved: ../Data/rag/rico_rag_dataset.csv


In [15]:
rag_jsonl_path = "../Data/rag/rico_rag_dataset.jsonl"

with open(rag_jsonl_path, "w", encoding="utf-8") as f:

    for _, row in rag_df.iterrows():

        record = {
            "id": row["record_id"],
            "text": row["text"],
            "metadata": {
                "record_type": row["record_type"],
                "title": row["title"],
                "source": row["source"],
            },
        }

        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Saved:", rag_jsonl_path)

Saved: ../Data/rag/rico_rag_dataset.jsonl


In [16]:
final_rag_df = pd.read_csv("../Data/rag/rico_rag_dataset.csv")

print("Final RAG dataset shape:", final_rag_df.shape)

print("\nRecord types:")
print(final_rag_df["record_type"].value_counts())

print("\nMissing values:")
print(final_rag_df.isna().sum())

print("\nDuplicate texts:")
print(final_rag_df["text"].duplicated().sum())

Final RAG dataset shape: (29, 5)

Record types:
record_type
spatial_relationship      16
component_cooccurrence     6
component_profile          4
cluster_profile            3
Name: count, dtype: int64

Missing values:
record_id      0
record_type    0
title          0
text           0
source         0
dtype: int64

Duplicate texts:
0
